In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, Annotated,List,Literal, Optional
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from dotenv import load_dotenv
import operator
from langchain_openai import ChatOpenAI
from langgraph.types import Send
from pydantic import BaseModel, Field
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph.message import add_messages

## SQL Tools

In [2]:
import sqlite3
from langchain.tools import tool

/Users/vatsal/Machine Learning/Gen AI/AIproject/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
@tool
def sql_db_list_tables() -> str:
    """Input is an empty string, output is a comma-separated list of tables in the database."""
    con = sqlite3.connect("/Users/vatsal/Machine Learning/Gen AI/AIproject/loan_data.db")
    try:
        cursor = con.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [row[0] for row in cursor.fetchall() if not row[0].startswith("sqlite_")]
        return ", ".join(tables)
    finally:
        con.close()

@tool
def sql_db_query(query: str) -> str:
    """Input to this tool is a detailed and correct SQL query, output is a result from the database.
    If the query is not correct, an error message will be returned.
    If an error is returned, rewrite the query, check the query, and try again."""
    # --- read-only guardrail: enforce BEFORE executing ---
    q = query.strip().lower()
    if not q.startswith("select"):
        return "Error: only SELECT queries are permitted."
    forbidden = ("insert", "update", "delete", "drop", "alter", "create", "replace", "truncate")
    if any(word in q.split() for word in forbidden):
        return "Error: query contains a forbidden keyword. Read-only access only."

    con = sqlite3.connect("/Users/vatsal/Machine Learning/Gen AI/AIproject/loan_data.db")
    try:
        cursor = con.cursor()
        cursor.execute(query)
        return str(cursor.fetchall())
    except Exception as e:
        return f"Error: {e}"
    finally:
        con.close()

@tool
def sql_db_query_checker(query: str) -> str:
    """Use this tool to double check if your query is correct before executing it.
    Always use this tool before executing a query with sql_db_query!"""
    trigger_prompt = """{query}
Double check the sqlite query above for common mistakes, including:
- Using NOT IN with NULL values
- Using UNION when UNION ALL should have been used
- Using BETWEEN for exclusive ranges
- Data type mismatch in predicates
- Properly quoting identifiers
- Using the correct number of arguments for functions
- Casting to the correct data type
- Using the proper columns for joins

If there are any of the above mistakes, rewrite the query. If there are no mistakes, just reproduce the original query.

Output the final SQL query only.

SQL Query: """.format(query=query)

    response = model.invoke(trigger_prompt)
    return response.text.strip()



In [ ]:

print(sql_db_list_tables.invoke(""))

print(sql_db_query.invoke("SELECT outstanding_balance FROM loans WHERE loan_id='L001'"))

print(sql_db_query.invoke("DELETE FROM loans WHERE loan_id='L001'"))

print(sql_db_query.invoke("SELECT COUNT(*) FROM loans"))

print(sql_db_query.invoke("SELECT * FROM nonexistent_table"))

customers, loans, repayments
[(240000.0,)]
Error: only SELECT queries are permitted.
[(5,)]
Error: no such table: nonexistent_table


## Loan Caluculation Tools

In [24]:
@tool
def monthly_payment(principal: float, annual_rate: float, term_months: int) -> str:
    """Calculates the monthly repayment for a loan given principal, annual rate, and term in months.
    USE WHEN: someone asks what the monthly payment is, or would be, for a given
    loan amount, rate, and term. Requires all three inputs — get them from the
    database first if they aren't provided in the question."""

    r = annual_rate / 12
    n = term_months
    result = principal / n if r == 0 else principal * (r*(1+r)**n) / ((1+r)**n - 1)
    return str(round(result, 2))

@tool
def lump_sum_recalc(outstanding_balance: float, annual_rate: float, remaining_months: int, lump_sum: float) -> str:
    """Recalculates the monthly payment after a one-off overpayment, keeping the term fixed.
    USE WHEN: someone asks how an overpayment or lump-sum payment would change their
    monthly payment. Requires current balance, rate, and remaining term — fetch these
    from the database for the specific loan before calling."""

    new_balance = outstanding_balance - lump_sum
    if new_balance <= 0:
        return {"new_monthly_payment": 0, "note": "loan fully repaid"}
    new_payment = float(monthly_payment.invoke({
    "principal": new_balance,
    "annual_rate": annual_rate,
    "term_months": remaining_months
}))
    return str({"new_balance": new_balance, "new_monthly_payment": round(new_payment, 2)})

@tool
def dti(total_monthly_debt: float, gross_monthly_income: float) -> str:
    """Calculates debt-to-income ratio as a percentage from total monthly debt and gross monthly income.
    USE WHEN: someone asks about affordability, or whether they can take on more debt.
    Requires their income and total monthly debt payments."""

    if gross_monthly_income <= 0:
        return {"error": "income must be positive"}
    return str(round((total_monthly_debt / gross_monthly_income) * 100, 2))

In [39]:
finacialtools = [sql_db_query_checker, sql_db_list_tables, sql_db_query, monthly_payment, dti, lump_sum_recalc ]

In [25]:
print(monthly_payment.invoke({
    "principal": 200000,
    "annual_rate": 0.04,
    "term_months": 300
}))

1055.67


In [26]:
print(lump_sum_recalc.invoke({
    "outstanding_balance": 240000,
    "annual_rate": 0.035,     # ← 3.5%, not 0.35
    "remaining_months": 264,
    "lump_sum": 20000
}))

{'new_balance': 220000.0, 'new_monthly_payment': 1196.1}


In [27]:
print(dti.invoke({
    "total_monthly_debt": 1500,
    "gross_monthly_income": 5000
}))

30.0


## RAG Tool

In [36]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(
    persist_directory="/Users/vatsal/Machine Learning/Gen AI/AIproject/chroma_db",
    embedding_function=embeddings,
)


@tool
def retrieve_doc(query: str, loan_id: str) -> str:
    """Retrieves relevant clauses from a specific customer's loan agreement.
    USE WHEN: the question is about what the CONTRACT permits, requires, or states —
    overpayment rules, penalties, early-repayment terms, payment-break eligibility,
    rate-change notice, missed-payment consequences. Requires the loan_id to filter
    to the correct customer's agreement.
    DO NOT use for account figures like current balance or interest rate — those come
    from the database, not the contract."""
    retriver = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 15, "filter": {"loan_id": loan_id}}
    )
    docs = retriver.invoke(query)
    if not docs:
        return "No relevant clauses found for this loan."
    return "\n\n".join(d.page_content for d in docs)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8531.41it/s]


In [37]:
# check: do the returned chunks' METADATA all say L003, even if content looks mixed?
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 15, "filter": {"loan_id": "L003"}}
)
docs = retriever.invoke("penalty for overpayment")
for d in docs:
    print(d.metadata.get("loan_id"), "|", d.metadata.get("source"), "|", d.page_content[:60])

L003 | L003_agreement.pdf | 1.2 Outstanding Balance & Duration: The present outstanding 
L003 | L003_agreement.pdf | made by the Borrower during the fixed-rate period shall incu
L003 | L003_agreement.pdf | CLAUSE 7. MISSED PAYMENTS, DEFAULT & ARREARS ENFORCEMENT
7.1


In [38]:
print(retrieve_doc.invoke({"query": "penalty for overpayment?", "loan_id": "L003"}))
print(retrieve_doc.invoke({"query": "penalty for overpayment?", "loan_id": "L001"}))

made by the Borrower during the fixed-rate period shall incur a mandatory financial penalty equal to 2.00% of the total
overpaid amount. This penalty fee is immediately payable upon execution of the overpayment. 
CLAUSE 5. EARLY REPAYMENT CHARGES (ERC)
5.1 Early Redemption Penalty: If the Borrower redeems or terminates the Loan agreement in full prior to the expiration of
the fixed-rate period, an Early Repayment Charge equal to three (3) months' gross interest on the total remaining principal
balance shall be assessed and added to the final redemption figure. 
CLAUSE 6. PROHIBITION OF PAYMENT BREAKS
6.1 Absolute Restriction: Payment breaks, payment holidays, or temporary moratoriums of any duration ARE NOT
PERMITTED under this agreement under any circumstances. The Borrower must maintain continuous uninterrupted
monthly repayments. 
Valut Bank Europe DAC — Confidential Mortgage Document Page 1 of 2

1.2 Outstanding Balance & Duration: The present outstanding principal balance is €190,